In [1]:
import os
import cv2
import numpy as np

# 1. Tentukan file input dan folder output
img_path = "materi_2.png"
output_dir = "dataset_asli"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Load gambar dan ubah ke Hitam-Putih murni (Binary Image)
img = cv2.imread(img_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Teks diubah jadi warna putih (255) dan background jadi hitam (0) agar mudah dihitung
_, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# 3. Hitung kerapatan horizontal (menghitung jumlah piksel putih dari kiri ke kanan)
horizontal_sum = np.sum(thresh, axis=1)

# Baris dianggap berisi teks jika jumlah piksel putihnya melebihi ambang batas tertentu
# (Diset minimal ada coretan sepanjang 10 piksel agar noise/bintik kecil tidak ikut terpotong)
has_text = horizontal_sum > 10

# 4. Algoritma Scanning untuk mencari batas atas dan batas bawah setiap kalimat
start_idx = None
count = 1

for y in range(len(has_text)):
    # Menemukan awal baris kalimat (transisi dari kosong ke ada tulisan)
    if has_text[y] and start_idx is None:
        start_idx = y
    # Menemukan akhir baris kalimat (transisi dari ada tulisan ke kosong kembali)
    elif not has_text[y] and start_idx is not None:
        end_idx = y

        # Pastikan tinggi potongan masuk akal (minimal tinggi baris 15 piksel)
        if end_idx - start_idx > 15:
            # Beri sedikit padding (jarak aman) atas-bawah sebesar 5 piksel agar huruf gak kepotong mepet
            y1 = max(0, start_idx - 5)
            y2 = min(img.shape[0], end_idx + 5)

            # Potong gambar secara horizontal dari ujung kiri ke ujung kanan
            cropped_line = img[y1:y2, 0 : img.shape[1]]

            # Simpan potongan gambar baris kalimat
            cv2.imwrite(f"{output_dir}/{count}.png", cropped_line)
            print(f"Berhasil memotong baris ke-{count} (Tinggi: {y2-y1}px)")
            count += 1

        start_idx = None  # Reset pencarian untuk baris selanjutnya

print(f"\nSelesai! Periksa folder '{output_dir}', semua potongan baris ada di sana.")

Berhasil memotong baris ke-1 (Tinggi: 57px)
Berhasil memotong baris ke-2 (Tinggi: 42px)
Berhasil memotong baris ke-3 (Tinggi: 53px)
Berhasil memotong baris ke-4 (Tinggi: 34px)
Berhasil memotong baris ke-5 (Tinggi: 34px)
Berhasil memotong baris ke-6 (Tinggi: 35px)
Berhasil memotong baris ke-7 (Tinggi: 41px)
Berhasil memotong baris ke-8 (Tinggi: 26px)
Berhasil memotong baris ke-9 (Tinggi: 33px)
Berhasil memotong baris ke-10 (Tinggi: 376px)
Berhasil memotong baris ke-11 (Tinggi: 26px)
Berhasil memotong baris ke-12 (Tinggi: 28px)
Berhasil memotong baris ke-13 (Tinggi: 35px)
Berhasil memotong baris ke-14 (Tinggi: 29px)
Berhasil memotong baris ke-15 (Tinggi: 33px)
Berhasil memotong baris ke-16 (Tinggi: 26px)

Selesai! Periksa folder 'dataset_asli', semua potongan baris ada di sana.


In [5]:
import os
import cv2
import numpy as np

img_path = "materi_2.png"
output_dir = "dataset_asli"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

img = cv2.imread(img_path)
if img is None:
    print(f"❌ EROR: File '{img_path}' tidak ditemukan!")
else:
    print(f"✅ Gambar termuat. Ukuran: {img.shape[1]}x{img.shape[0]} px.")

    # 1. Ubah ke Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. Tambahkan Blur untuk menghapus noise halus di sela-sela baris
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # 3. Adaptive Thresholding (Kita bikin lebih ketat dengan menaikkan konstanta ke 15)
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 25, 15
    )

    # 4. Pembersihan Total (Menghapus bintik-bintik putih terisolasi yang bukan tulisan)
    kernel = np.ones((3, 3), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    # 5. Hitung kerapatan horizontal
    horizontal_sum = np.sum(thresh, axis=1)

    # UPGRADE: Naikkan batas minimal menjadi 5% dari lebar gambar (biar gak gampang kegabung)
    min_thresh_value = int(img.shape[1] * 0.05 * 255)
    has_text = horizontal_sum > min_thresh_value

    print(
        f"📊 Debug: Mengunci area tulisan setelah saringan diperketat..."
    )

    # 6. Proses Pemotongan Per Baris
    start_idx = None
    count = 1

    for y in range(len(has_text)):
        if has_text[y] and start_idx is None:
            start_idx = y
        elif not has_text[y] and start_idx is not None:
            end_idx = y

            # Syarat tinggi baris (minimal 20 piksel biar gak motong potongan compang-camping)
            if end_idx - start_idx > 20:
                # Padding atas-bawah kasih sedikit jarak aman (3 piksel)
                y1 = max(0, start_idx - 3)
                y2 = min(img.shape[0], end_idx + 3)

                cropped_line = img[y1:y2, 0 : img.shape[1]]
                cv2.imwrite(f"{output_dir}/{count}.png", cropped_line)
                print(f"✂️ Berhasil memotong baris ke-{count} (Tinggi: {y2-y1}px)")
                count += 1

            start_idx = None

    # Sisa baris paling bawah kertas jika ada
    if start_idx is not None and (img.shape[0] - start_idx > 20):
        y1 = max(0, start_idx - 3)
        y2 = img.shape[0]
        cropped_line = img[y1:y2, 0 : img.shape[1]]
        cv2.imwrite(f"{output_dir}/{count}.png", cropped_line)
        count += 1

    print(f"\n🎉 SELESAI! Total baris murni yang didapat: {count - 1} file.")

✅ Gambar termuat. Ukuran: 721x961 px.
📊 Debug: Mengunci area tulisan setelah saringan diperketat...

🎉 SELESAI! Total baris murni yang didapat: 0 file.


In [9]:
import os
import cv2
import numpy as np

img_path = "materi_2.png"  # Sesuaikan dengan nama file lu
output_dir = "dataset_asli"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

img = cv2.imread(img_path)
if img is None:
    print(f"❌ EROR: File '{img_path}' tidak ditemukan!")
else:
    print(f"✅ Gambar termuat. Ukuran: {img.shape[1]}x{img.shape[0]} px.")

    # 1. Grayscale & Blur ringan
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # 2. Binarization (Adaptive Thresholding)
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 51, 4
    )

    # 3. TRIK ANTI-GABUNG: Bersihkan margin kiri (15% lebar kertas)
    # Ini menghapus nomor list/garis tepi yang bikin baris-baris kalimat gandengan vertikal
    margin_left = int(img.shape[1] * 0.15)
    thresh[:, 0:margin_left] = 0 

    # 4. JURUS STACKOVERFLOW: Dilasi Horizontal (Membuat kata-kata lumer jadi satu baris)
    # Menggunakan kernel horizontal lebar (lebar 100 piksel, tinggi 1 piksel)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (100, 1))
    dilated = cv2.dilate(thresh, kernel, iterations=1)

    # 5. JURUS STACKOVERFLOW: Deteksi Kontur Objek
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 6. SORTING: Urutkan kontur dari atas ke bawah berdasarkan koordinat Y
    contours = sorted(contours, key=lambda ctr: cv2.boundingRect(ctr)[1])

    print(f"📊 Debug: Menghitung baris menggunakan metode Kontur StackOverflow...")

    # 7. Proses Crop berdasarkan Bounding Box Kontur
    count = 1
    for ctr in contours:
        # Ambil koordinat kotak pembungkus teks
        x, y, w, h = cv2.boundingRect(ctr)

        # FILTER: Hanya ambil yang beneran baris kalimat 
        # (Lebar harus > 150px dan Tinggi > 20px biar coretan sampah gak ikut ke-crop)
        if w > 150 and h > 20:
            
            # Kasih sedikit padding aman (5 piksel) biar tulisan gak kepotong mepet
            y1 = max(0, y - 5)
            y2 = min(img.shape[0], y + h + 5)
            
            # Kerennya metode ini: Kita bisa potong pas di teksnya (x sampai x+w)
            # Tapi karena ini teks binder, kita ambil agak ke kiri dikit (x-30) 
            # supaya nomor list yang tadi kita sembunyikan TETAP IKUT KEBAWAH di hasil crop akhir!
            x1 = max(0, x - 40)
            x2 = min(img.shape[1], x + w + 10)

            # Eksekusi Potong
            cropped_line = img[y1:y2, x1:x2]

            # Simpan hasil
            cv2.imwrite(f"{output_dir}/{count}.png", cropped_line)
            print(f"✂️ Baris {count} Berhasil! -> Ukuran Kotak: {w}x{h} px")
            count += 1

    print(f"\n🎉 SELESAI SAKSES! Total murni dapet: {count - 1} baris kalimat.")

✅ Gambar termuat. Ukuran: 721x961 px.
📊 Debug: Menghitung baris menggunakan metode Kontur StackOverflow...
✂️ Baris 1 Berhasil! -> Ukuran Kotak: 662x30 px
✂️ Baris 2 Berhasil! -> Ukuran Kotak: 662x23 px
✂️ Baris 3 Berhasil! -> Ukuran Kotak: 662x42 px
✂️ Baris 4 Berhasil! -> Ukuran Kotak: 662x22 px
✂️ Baris 5 Berhasil! -> Ukuran Kotak: 662x54 px
✂️ Baris 6 Berhasil! -> Ukuran Kotak: 662x25 px
✂️ Baris 7 Berhasil! -> Ukuran Kotak: 662x32 px
✂️ Baris 8 Berhasil! -> Ukuran Kotak: 662x52 px
✂️ Baris 9 Berhasil! -> Ukuran Kotak: 662x23 px
✂️ Baris 10 Berhasil! -> Ukuran Kotak: 662x23 px
✂️ Baris 11 Berhasil! -> Ukuran Kotak: 662x40 px
✂️ Baris 12 Berhasil! -> Ukuran Kotak: 662x56 px
✂️ Baris 13 Berhasil! -> Ukuran Kotak: 662x25 px
✂️ Baris 14 Berhasil! -> Ukuran Kotak: 662x26 px

🎉 SELESAI SAKSES! Total murni dapet: 14 baris kalimat.


In [11]:
"""
segmentasi_baris.py
====================
Ekstrak baris tulisan tangan dari foto kertas bergaris.

Cara pakai:
    python segmentasi_baris.py --input foto.png --output folder_hasil

Opsional:
    --min_ink 0.04       sensitivitas peak (lebih kecil = lebih banyak baris)
    --padding 3          pixel padding atas-bawah tiap crop
    --min_cc 2           jumlah minimum connected component per baris (filter noise)
    --min_cc_area 30     ukuran minimum (px) tiap connected component

Dependensi:
    pip install opencv-python-headless numpy scipy
"""

import argparse
import os
import cv2
import numpy as np
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks, savgol_filter


def segmentasi_baris(
    img_path: str,
    output_dir: str,
    min_ink: float = 0.04,
    padding: int = 3,
    min_cc: int = 2,
    min_cc_area: int = 30,
):
    # ── 1. Baca gambar ──────────────────────────────────────────────────────
    img = cv2.imread(img_path)
    if img is None:
        print(f"❌ Tidak bisa membaca: {img_path}")
        return []

    H, W = img.shape[:2]
    print(f"✅ Gambar: {W}x{H} px")

    # ── 2. Pre-processing ───────────────────────────────────────────────────
    gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    binary  = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        31, 8
    )

    # ── 3. Hapus garis horizontal kertas bergaris ────────────────────────────
    # Garis kertas = struktur horizontal ≥ 65% lebar gambar
    kernel_line    = cv2.getStructuringElement(cv2.MORPH_RECT, (int(W * 0.65), 1))
    horiz_lines    = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_line, iterations=1)
    # Sedikit dilasi vertikal agar garis terhapus sempurna
    kernel_dv      = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 3))
    horiz_lines_d  = cv2.dilate(horiz_lines, kernel_dv)
    binary_clean   = cv2.subtract(binary, horiz_lines_d)

    # ── 4. Proyeksi horizontal + smoothing ──────────────────────────────────
    row_sums   = np.sum(binary_clean, axis=1) / 255.0
    row_smooth = uniform_filter1d(row_sums.astype(float), size=3)
    row_savgol = savgol_filter(row_smooth, window_length=7, polyorder=2)
    row_savgol = np.clip(row_savgol, 0, None)

    max_val  = row_savgol.max()
    if max_val == 0:
        print("❌ Gambar kosong — tidak ada tinta terdeteksi.")
        return []
    row_norm = row_savgol / max_val

    # ── 5. Deteksi puncak (pusat tiap baris teks) ────────────────────────────
    peaks, _ = find_peaks(
        row_norm,
        height=min_ink,
        distance=10,
        prominence=min_ink
    )

    if len(peaks) == 0:
        print("⚠️  Tidak ada baris terdeteksi. Coba turunkan --min_ink")
        return []

    lh = int(np.median(np.diff(peaks))) if len(peaks) >= 2 else 20
    print(f"📏 Estimasi tinggi baris: {lh}px | Puncak: {len(peaks)}")

    # ── 6. Tentukan batas baris dari valley antara puncak ───────────────────
    separators = [0]
    for i in range(len(peaks) - 1):
        seg         = row_norm[peaks[i]: peaks[i + 1]]
        valley_abs  = peaks[i] + int(np.argmin(seg))
        separators.append(valley_abs)
    separators.append(H - 1)

    # ── 7. Filter: buang segmen yang tidak punya cukup connected component ──
    segments = []
    for i in range(len(separators) - 1):
        y1, y2   = separators[i], separators[i + 1]
        seg_ink  = row_sums[y1: y2 + 1]
        nonzero  = np.where(seg_ink > 2)[0]

        if len(nonzero) == 0:
            continue

        # Trim tepi kosong + padding
        ty1 = max(0,   y1 + nonzero[0]  - padding)
        ty2 = min(H-1, y1 + nonzero[-1] + padding)
        if (ty2 - ty1) < 5:
            continue

        # Hitung connected component yang cukup besar (bukan noise / garis)
        crop_bin   = binary_clean[ty1: ty2 + 1, :]
        num_labels, _, stats, _ = cv2.connectedComponentsWithStats(crop_bin, connectivity=8)
        big_cc = sum(
            1 for j in range(1, num_labels)
            if stats[j, cv2.CC_STAT_AREA] >= min_cc_area
        )

        if big_cc >= min_cc:
            segments.append((ty1, ty2))

    # ── 8. Simpan crop ───────────────────────────────────────────────────────
    os.makedirs(output_dir, exist_ok=True)
    for idx, (y1, y2) in enumerate(segments):
        crop     = img[y1: y2 + 1, :, :]
        filename = os.path.join(output_dir, f"{idx + 1:03d}.png")
        cv2.imwrite(filename, crop)

    print(f"\n🎉 Selesai! {len(segments)} baris disimpan ke '{output_dir}/'")
    return segments


# ── CLI ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Segmentasi baris tulisan tangan")
    parser.add_argument("--input",       required=True,            help="Path gambar input")
    parser.add_argument("--output",      default="output_baris",   help="Folder output")
    parser.add_argument("--min_ink",     type=float, default=0.04, help="Sensitivitas (default: 0.04)")
    parser.add_argument("--padding",     type=int,   default=3,    help="Padding piksel (default: 3)")
    parser.add_argument("--min_cc",      type=int,   default=2,    help="Min. connected component (default: 2)")
    parser.add_argument("--min_cc_area", type=int,   default=30,   help="Min. area CC dalam px (default: 30)")
    args = parser.parse_args()

    segmentasi_baris(
        img_path    = args.input,
        output_dir  = args.output,
        min_ink     = args.min_ink,
        padding     = args.padding,
        min_cc      = args.min_cc,
        min_cc_area = args.min_cc_area,
    )

usage: ipykernel_launcher.py [-h] --input INPUT [--output OUTPUT] [--min_ink MIN_INK] [--padding PADDING]
                             [--min_cc MIN_CC] [--min_cc_area MIN_CC_AREA]
ipykernel_launcher.py: error: the following arguments are required: --input


SystemExit: 2

D:\Miniconda3\envs\ocr\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
